# Côte d'Ivoire - Projet PROMIRE / ONU-REDD

1 · Calcul de l'activité 2021-2025 à partir de l'occupation du sol interprétée dans CEO
-------
Ce notebook attribue à chaque échantillon une activité (SNF, SF, SDegF, Deg, Def, Gain) en comparant les classes d'occupation du sol de 2021 et de 2025 interprétées dans Collect Earth Online (CEO), aux niveaux 1 et 2 de la nomenclature nationale. Les critères suivent ceux du NERF 2024, complétés par une matrice de transition pour les changements entre classes forestières.

**Entrées** (dossier `data/`) : l'export CEO des échantillons interprétés, la strate de chaque échantillon et la limite du PRE.

**Sortie** : `results/CIV_echantillons_activite_2021_2025.csv`, utilisé par les notebooks `2` et `3` : une ligne par échantillon avec `plotid`, `center_lon`, `center_lat`, `strata`, `pre` (échantillon situé dans le PRE), `activity`, et les classes de chaque année.

In [ ]:
import os

import numpy as np
import pandas as pd
import geopandas as gpd

## 1. Lecture des données

In [ ]:
points_file   = 'data/ceo-RCI_MONITORING_21-25-plot-data_QA_QC-rev-2.csv'   # export CEO des échantillons interprétés
points_strata = 'data/allWithStrata_civ_20_26.shp'                         # strate de chaque échantillon (colonne mode)
pre_file      = 'data/shapefile_PRE_repaired_geometry.shp'                 # limite du PRE

output_file   = 'results/CIV_echantillons_activite_2021_2025.csv'

annee_initiale = 2021
annee_finale   = 2025
annees = list(range(annee_initiale, annee_finale + 1))

In [ ]:
# Échantillons interprétés dans CEO
df = pd.read_csv(points_file, encoding='latin-1')
print(len(df), 'échantillons')

### Strate de chaque échantillon

Les points des strates sont les échantillons eux-mêmes : on cherche le point le plus proche, dans une projection métrique (UTM 30N), et la distance doit être nulle.

In [ ]:
strata = gpd.read_file(points_strata).rename(columns={'mode': 'strata'})
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.center_lon, df.center_lat), crs='EPSG:4326')
gdf = gpd.sjoin_nearest(gdf.to_crs('EPSG:32630'), strata[['strata', 'geometry']].to_crs('EPSG:32630'),
                        how='left', distance_col='distance_m')
print('Distance maximale au point de strate :', gdf['distance_m'].max(), 'm')

df = pd.DataFrame(gdf.drop(columns=['geometry', 'index_right', 'distance_m']))
assert df.plotid.is_unique, 'plotid en double'
df.strata.value_counts(dropna=False)

Les échantillons sans strate (hors de la frontière nationale ou dans une zone masquée de la carte de stratification) sont conservés ici ; les notebooks d'estimation les excluent.

### Échantillons situés dans le PRE

Le Programme de Réductions des Émissions (PRE) couvre une partie du territoire. La colonne `pre` vaut `True` si l'échantillon se trouve dans le polygone du PRE (limite comprise) et `False` sinon. Les notebooks `2` et `3` s'en servent pour produire les estimations du PRE.

In [ ]:
pre = gpd.read_file(pre_file).to_crs('EPSG:32630')
points = gpd.GeoSeries(gpd.points_from_xy(df.center_lon, df.center_lat), crs='EPSG:4326').to_crs('EPSG:32630')
df['pre'] = points.intersects(pre.union_all()).to_numpy()
print(df['pre'].sum(), 'échantillons dans le PRE sur', len(df))
pd.crosstab(df['pre'], df['strata'].fillna(0).astype(int).rename('strate (0 = sans strate)'), margins=True)

## 2. Classes d'occupation du sol par année et par niveau

Dans le fichier CEO, chaque classe est une colonne (par exemple `Occupation_2021:Terres_Forestieres` ou `TF_Niveau2_2021:Foret_Dense`) qui vaut 100 pour la classe choisie par l'interprète et 0 sinon. On construit trois tableaux, un par niveau, avec une ligne par échantillon et une colonne par année.

In [ ]:
# Colonnes d'occupation du sol (on écarte les réponses sur le changement, les moteurs et les années d'événement)
colonnes_ocs = [c for c in df.columns
                if '_20' in c and not c.startswith(('Changement', 'Moteur', 'Annee', 'Gain'))]

# Format long : une ligne par échantillon et par classe choisie
ocs = df.melt(id_vars='plotid', value_vars=colonnes_ocs, var_name='colonne', value_name='valeur')
ocs = ocs[ocs.valeur == 100].copy()
ocs[['variable', 'classe']] = ocs.colonne.str.split(':', expand=True)
ocs['annee'] = ocs.variable.str[-4:].astype(int)
ocs['niveau'] = np.select(
    [ocs.variable.str.startswith('Occupation'), ocs.variable.str.contains('Niveau2'), ocs.variable.str.contains('Niveau3')],
    [1, 2, 3], default=0)
assert (ocs.niveau > 0).all()
ocs.head()

In [ ]:
def classes_par_annee(niveau):
    """Tableau échantillons × années des classes d'un niveau (erreur si un échantillon a deux classes la même année)."""
    tableau = ocs[ocs.niveau == niveau].pivot(index='plotid', columns='annee', values='classe')
    return tableau.reindex(index=df.plotid, columns=annees)


n1 = classes_par_annee(1)
n2 = classes_par_annee(2)
n3 = classes_par_annee(3)
n1.head()

## 3. Échantillons sans classe de niveau 1

Une classe de niveau 1 absente ou égale à `no_data` empêche de savoir si l'échantillon est forestier. Seules les années 2021 et 2025 servent au calcul de l'activité : un échantillon sans classe l'une de ces deux années reçoit l'activité `ND` et sera exclu des estimations.

In [ ]:
manquant = n1.isna() | (n1 == 'no_data')
print('Nombre d\'échantillons sans classe de niveau 1, par année :')
print(manquant.sum().to_string())

sans_classe = manquant[manquant.any(axis=1)]
liste = pd.DataFrame({
    'plotid': sans_classe.index,
    'annees_sans_classe': [', '.join(str(a) for a in annees if ligne[a]) for _, ligne in sans_classe.iterrows()],
    'activite_ND': (sans_classe[annee_initiale] | sans_classe[annee_finale]).values,
})
liste

## 4. Calcul de l'activité

Critères du NERF 2024, complétés par la matrice de transition 2021-2025, appliqués aux années 2021 (initiale) et 2025 (finale). « Forêt » = classe de niveau 1 `Terres_Forestieres`.

**Changements à l'intérieur de la forêt.** Chaque classe forestière de niveau 2 est rattachée à une classe de la matrice :

| Classe de la matrice | Classes de niveau 2 (CEO) |
|:--|:--|
| 11. Forêt Dense | `Foret_Dense`, `Foret_Galerie`, `Foret_Sol_Hydromorphe`, `Mangrove` |
| 12. Forêt Secondaire | `Foret_Secondaire_Degradee` |
| 13. Plantations forestières | `Plantation_Forestiere_Reboisement` |
| 14. Agroforesterie | `Plantation_Agroforestiere` |

Le cacao, y compris le cacao sous ombrage, reste une classe cultivée (non-forêt). La matrice ordonne les classes 11 > 12 > 13 > 14 :

| 2021 ↓ \ 2025 → | 11 | 12 | 13 | 14 |
|:--|:--|:--|:--|:--|
| **11** | SF | Deg | Def | Deg |
| **12** | Gain | SDegF | Def | Deg |
| **13** | Gain | Gain | SF | Deg |
| **14** | Gain | Gain | Gain | SDegF |

Une forêt qui passe à une classe de rang inférieur est dégradée, sauf vers une plantation forestière (déforestation). Une forêt qui passe à une classe de rang supérieur est un gain. Comme dans le NERF 2024, les forêts stables secondaires et agroforestières sont classées SDegF.

**Règles.** Elles s'excluent mutuellement :

| Ordre | Activité | Règle |
|:--|:--|:--|
| 1 | ND | classe de niveau 1 manquante en 2021 ou en 2025 |
| 2 | Def | forêt en 2021, non-forêt en 2025 ; ou forêt dense / secondaire en 2021 et plantation forestière en 2025 |
| 3 | Deg | forêt en 2021 et en 2025, classe de rang inférieur en 2025 (sauf plantation forestière) |
| 4 | Gain | non-forêt en 2021, forêt en 2025 ; ou forêt en 2021 et en 2025, classe de rang supérieur en 2025 |
| 5 | SDegF | forêt secondaire en 2021 et en 2025, ou agroforêt en 2021 et en 2025 |
| 6 | SF | forêt en 2021 et en 2025, même classe de la matrice (11 ou 13) |
| 7 | SNF | non-forêt en 2021 et en 2025 |

Gain regroupe donc deux cas : non-forêt → forêt, et forêt → forêt de rang supérieur. On peut les séparer dans le notebook des émissions avec `n1_2021`.

In [ ]:
foret = 'Terres_Forestieres'

# Classe de la matrice de transition pour chaque classe forestière de niveau 2 (rang : 11 > 12 > 13 > 14)
classe_matrice = {
    'Foret_Dense': 11, 'Foret_Galerie': 11, 'Foret_Sol_Hydromorphe': 11, 'Mangrove': 11,
    'Foret_Secondaire_Degradee': 12,
    'Plantation_Forestiere_Reboisement': 13,
    'Plantation_Agroforestiere': 14,
}

# Les noms de classes doivent exister dans les données (protection contre les fautes de frappe)
assert foret in set(n1.stack())
assert set(classe_matrice) <= set(n2.stack())

In [ ]:
# Classes de 2021 (i = initiale) et de 2025 (f = finale)
n1_i, n1_f = n1[annee_initiale], n1[annee_finale]
n2_i, n2_f = n2[annee_initiale], n2[annee_finale]

foret_i = n1_i == foret
foret_f = n1_f == foret
reste_foret = foret_i & foret_f

# Classe de la matrice (11 à 14) des échantillons forestiers, NaN sinon
assert set(n2_i[foret_i]) | set(n2_f[foret_f]) <= set(classe_matrice), 'classe forestière sans correspondance dans la matrice'
m_i = n2_i.map(classe_matrice).where(foret_i)
m_f = n2_f.map(classe_matrice).where(foret_f)

classe_manquante = manquant[annee_initiale] | manquant[annee_finale]
deforestation  = (foret_i & ~foret_f) | (m_i.isin([11, 12]) & (m_f == 13))
degradation    = reste_foret & (m_f > m_i) & ~deforestation
gain           = (~foret_i & foret_f) | (reste_foret & (m_f < m_i))
stable_degrade = reste_foret & (m_i == m_f) & m_i.isin([12, 14])

conditions = [
    classe_manquante,
    deforestation,
    degradation,
    gain,
    stable_degrade,
    reste_foret,
    ~foret_i & ~foret_f,
]
activites = ['ND', 'Def', 'Deg', 'Gain', 'SDegF', 'SF', 'SNF']

activity = pd.Series(np.select(conditions, activites, default='Erreur'), index=n1.index, name='activity')
assert (activity != 'Erreur').all()
activity.value_counts()

In [ ]:
# Classes de niveau 2 des échantillons forestiers en 2021
n2_i[foret_i].value_counts()

In [ ]:
# Contrôle : activité selon la classe de niveau 1 en 2021
pd.crosstab(n1_i.fillna('(manquante)').rename('niveau 1 en 2021'), activity)

In [ ]:
# Contrôle : activité des échantillons forestiers en 2021 et en 2025, selon la classe de la matrice (à comparer à la matrice de transition)
pd.crosstab([m_i[reste_foret].astype(int).rename('matrice 2021'), m_f[reste_foret].astype(int).rename('matrice 2025')],
            activity[reste_foret])

## 5. Enregistrement

In [ ]:
sortie = df.set_index('plotid')[['center_lon', 'center_lat', 'strata', 'pre']]
sortie['activity'] = activity
sortie = (sortie.join(n1.add_prefix('n1_'))
                .join(n2.add_prefix('n2_'))
                .join(n3[[annee_initiale, annee_finale]].add_prefix('n3_'))
                .reset_index())

assert len(sortie) == len(df)
os.makedirs(os.path.dirname(output_file), exist_ok=True)
sortie.to_csv(output_file, index=False)
print(len(sortie), 'échantillons enregistrés dans', output_file)
sortie.activity.value_counts()